# EX-10 Flight · real footage in the dome

**A film · Python + numpy + OpenCV · green-screen footage keyed and flown around the room**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dcuartielles/wisdome-360/blob/main/docs/examples/ex10-flight.ipynb)

Something filmed on green screen circles the audience under a dark dusk sky, once every thirty seconds. Its picture isn't generated: it is video, keyed and hung on a card in the room. This is the example for "can we use our own video?"

![Frames of Flight with the test clip: a white shape keyed from green, flying in front of the audience](https://dcuartielles.github.io/wisdome-360/examples/images/ex10-flight.jpg)

| File | What it is |
|---|---|
| This notebook | The method, step by step. Runs on a green-screen **test clip** it makes itself, or on your own footage |
| [`flight.py`](https://dcuartielles.github.io/wisdome-360/examples/files/EX-10-greenscreen-footage/flight.py) | The same piece as one script, set up for the dove clip below: `python3 flight.py` writes the master and the keyed bird with transparency |

### The dove clip

The original piece flies a white dove: ["Flying Dove – Free Green screen Stock Footage"](https://ultrakey.gumroad.com/l/flyingdove) by **UltraKey**. It is free for non-commercial use with credit to UltraKey; client or monetised work needs their paid licence. Its licence doesn't let us share it, or the films made from it, so **download it from UltraKey yourself**, then either save it as `source/dove_greenscreen.mkv` next to `flight.py`, or upload it to this notebook (below).

In [ ]:
%pip install -q numpy opencv-python-headless

In [ ]:
import math, os, subprocess
import numpy as np
import cv2
from PIL import Image
from IPython.display import display

## Settings

Design at 1024; 2048 is the [delivery size](https://dcuartielles.github.io/wisdome-360/reference/delivery-spec/).

In [ ]:
RES      = 1024         # 2048 for delivery
FPS      = 30
SECONDS  = 30           # one lap of the room
TILT     = 26.0         # dome tilt in degrees, back raised (Wisdome Malmö)
BIRD_DEG = 30.0         # angular width of the card the footage hangs on

SOURCE   = None         # None = the test clip; or a video file, e.g. "dove_greenscreen.mkv"
CROP     = (560, 200, 1352, 880)   # x0, y0, x1, y1: a box round the subject in the 1920x1080 source

**Your own footage.** In Colab, set `UPLOAD = True` and run the cell; in Jupyter, put the file next to the notebook and set `SOURCE` above. Shoot the subject roughly in place in the frame, like a bird flapping on the spot: the flight comes from `flight_path()`, not from the camera. Then set `CROP` to a box around it.

In [ ]:
UPLOAD = False

if UPLOAD:
    from google.colab import files
    SOURCE = next(iter(files.upload()))

## The footage

`frames()` gives 1920 × 1080 RGB frames from `SOURCE`, looping if it is shorter than the piece. Without a source, it draws a test clip: a white bird shape flapping on the spot, on a green backing.

In [ ]:
def test_frame(i, w=1920, h=1080):
    """A green-screen test frame: a white shape facing right, flapping 3 times a second."""
    img = np.zeros((h, w, 3), np.uint8); img[:] = (40, 180, 70)          # green backing (RGB)
    cx, cy = 956, 560
    flap = math.sin(2 * math.pi * 3 * i / FPS)
    white, shade = (245, 245, 240), (205, 210, 215)
    for k, colour in ((0, shade), (1, white)):                             # far wing, near wing
        ty = int(cy - 30 - (280 - 60 * k) * flap)                          # wing tip height
        lift = int(110 * (1 - abs(flap)))                                  # edge-on wings still show
        pts = np.array([(cx - 120 + 30 * k, cy - 20), (cx + 70 + 30 * k, cy - 20),
                        (cx + 10 + 30 * k, ty - lift), (cx - 170 + 30 * k, ty - lift + 50)], np.int32)
        cv2.fillPoly(img, [pts], colour, cv2.LINE_AA)
    cv2.ellipse(img, (cx, cy), (190, 62), 0, 0, 360, white, -1, cv2.LINE_AA)   # body
    cv2.circle(img, (cx + 190, cy - 30), 48, white, -1, cv2.LINE_AA)            # head
    cv2.fillPoly(img, [np.array([(cx + 232, cy - 40), (cx + 290, cy - 22), (cx + 232, cy - 12)], np.int32)],
                 (240, 170, 60), cv2.LINE_AA)                                    # beak
    cv2.fillPoly(img, [np.array([(cx - 170, cy), (cx - 300, cy - 60), (cx - 300, cy + 60)], np.int32)],
                 white, cv2.LINE_AA)                                             # tail
    return img


def frames(n):
    if SOURCE is None:
        for i in range(n):
            yield test_frame(i)
        return
    cap = cv2.VideoCapture(SOURCE)
    for i in range(n):
        ok, bgr = cap.read()
        if not ok:                                   # loop the clip
            cap.set(cv2.CAP_PROP_POS_FRAMES, 0); ok, bgr = cap.read()
        yield cv2.resize(bgr, (1920, 1080))[..., ::-1]

first = next(frames(1))
x0, y0, x1, y1 = CROP
preview = first.copy(); cv2.rectangle(preview, (x0, y0), (x1, y1), (255, 60, 60), 6)
display(Image.fromarray(preview).resize((640, 360)))

## 1 · Key: `key()`

For every pixel, "spill" is how much greener it is than its other two channels: about 1 on the backing, about 0 on white feathers. A soft ramp between the two becomes the alpha matte. Then **despill**: clamp green down to the brighter of red and blue. Skip that, and every edge keeps a green fringe, which is the first thing people see against a dark dome sky.

In [ ]:
def key(rgb):
    """Green-screen key. uint8 HxWx3 -> float RGBA in 0..1."""
    f = rgb.astype(np.float32) / 255.0
    r, g, b = f[..., 0], f[..., 1], f[..., 2]
    spill = g - np.maximum(r, b)
    a = 1.0 - np.clip((spill - 0.04) / 0.30, 0, 1)
    a = cv2.GaussianBlur(a, (3, 3), 0)
    a[a < 0.02] = 0
    g = np.minimum(g, np.maximum(r, b) + 0.02)        # despill
    return np.dstack([r, g, b, a])

bird = key(first[y0:y1, x0:x1])
check = (((np.indices(bird.shape[:2]) // 24).sum(0) % 2) * 0.25 + 0.2)[..., None]   # a checkerboard
over = bird[..., :3] * bird[..., 3:4] + check * (1 - bird[..., 3:4])
display(Image.fromarray((over * 255).astype(np.uint8)).resize((396, 340)))

## The one place the 26° tilt lives

Each pixel of the master becomes a direction in the room. Room axes: x = right, y = front, z = up. Azimuth 0 is the front, elevation 0 is eye level, as in every other example.

In [ ]:
def master_to_room(n):
    yy, xx = np.mgrid[0:n, 0:n].astype(np.float32)
    c = (n - 1) / 2.0
    dx, dy = xx - c, yy - c
    r = np.sqrt(dx * dx + dy * dy) / (n / 2.0)
    inside = r <= 1.0
    th = np.radians(90.0) * np.clip(r, 0, 1)
    a = np.arctan2(dx, dy)                                  # 0 = bottom = front
    lx, ly, lz = np.sin(th) * np.sin(a), np.sin(th) * np.cos(a), np.cos(th)
    ct, st = math.cos(math.radians(TILT)), math.sin(math.radians(TILT))
    room = np.stack([lx, ly * ct + lz * st, -ly * st + lz * ct], -1).astype(np.float32)
    return room, inside


def room_to_master(d):
    """The inverse, for a list of room directions -> pixel xy."""
    ct, st = math.cos(math.radians(TILT)), math.sin(math.radians(TILT))
    x, y, z = d[:, 0], d[:, 1], d[:, 2]
    lx, ly, lz = x, y * ct - z * st, y * st + z * ct
    th = np.arccos(np.clip(lz, -1, 1)); a = np.arctan2(lx, ly)
    r = th / (np.pi / 2) * (RES / 2.0); c = (RES - 1) / 2.0
    xy = np.stack([c + r * np.sin(a), c + r * np.cos(a)], -1)
    return xy, (th <= np.pi / 2) & (xy.min(1) >= 0) & (xy.max(1) < RES)


def room_dir(az, el):
    a, e = math.radians(az), math.radians(el)
    return np.array([math.cos(e) * math.sin(a), math.cos(e) * math.cos(a), math.sin(e)], np.float32)


ROOM, INSIDE = master_to_room(RES)
EL = np.degrees(np.arcsin(np.clip(ROOM[..., 2], -1, 1)))
AZ = np.degrees(np.arctan2(ROOM[..., 0], ROOM[..., 1]))      # -180..180, 0 = front

## The sky

Near-black, because the dome cross-reflects and a bright sky would grey out a white subject. A warm glow low at the front and cool behind gives the room a direction. Stars are texture at the top, scattered with `arcsin` so they don't clump at the zenith.

In [ ]:
def make_sky():
    front = 0.5 + 0.5 * np.cos(np.radians(AZ))                 # 1 in front, 0 behind
    low = np.exp(-np.clip(EL, 0, 90) / 14.0)                   # strongest at eye level
    base = np.array([0.010, 0.014, 0.030], np.float32)
    warm = np.array([0.30, 0.12, 0.035], np.float32)
    cool = np.array([0.025, 0.05, 0.11], np.float32)
    sky = base + (low * front ** 2)[..., None] * warm + (low * (1 - front) * 0.8)[..., None] * cool
    rng = np.random.default_rng(10)
    m = 2600
    saz = rng.uniform(-180, 180, m)
    sel = np.degrees(np.arcsin(rng.uniform(math.sin(math.radians(10)), 1, m)))
    sbr = rng.uniform(0.25, 0.9, m) ** 2
    stars = np.zeros(sky.shape[:2], np.float32)
    dirs = np.stack([np.cos(np.radians(sel)) * np.sin(np.radians(saz)),
                     np.cos(np.radians(sel)) * np.cos(np.radians(saz)),
                     np.sin(np.radians(sel))], -1)
    px, ok = room_to_master(dirs)
    for (x, y), b, k in zip(px, sbr, ok):
        if k:
            stars[int(y), int(x)] += b
    stars = cv2.GaussianBlur(stars, (0, 0), RES / 2048 * 0.8)
    stars *= (1 - low * front) * 6.0                           # fade into the glow
    return sky + stars[..., None] * np.array([0.9, 0.93, 1.0], np.float32)

SKY = make_sky()

## 2 · Fly: `flight_path()`

The card moves in **(azimuth, elevation)**. One lap is 360° in 30 s, 12° a second.

| Where | Elevation | Why |
|---|---|---|
| In front (azimuth 0) | about 30° | inside the comfortable zone, close to the 40° sweet spot |
| Behind (azimuth 180) | about 48° | behind the audience the picture only starts about 36° up; any lower and it flies out of the dome |
| All the way round | ±3° bob, every 6 s | a level circle reads as a turntable; a small bob reads as flight |

In [ ]:
def flight_path(t):
    az = -60.0 + 360.0 * t / SECONDS
    back = 0.5 - 0.5 * math.cos(math.radians(az))              # 0 in front, 1 behind
    el = 30.0 + 18.0 * back + 3.0 * math.sin(2 * math.pi * t / 6.0)
    return az, el

## 3 · Place: `place_bird()`

The keyed frame hangs on a flat card out in the room, facing the audience. Its right edge points along the flight path (the subject faces right in the source), and its top edge points as close to the room's up as it can, which keeps the roll at zero. Each master pixel's ray meets the card, and where it lands is the source pixel it sees.

> Never paste flat footage into the dome master. Put it on a card **in the room** and let the projection do the bending.

Pasted straight onto the master, the clip looks right in the middle and wrong everywhere else: it stretches and curls towards the rim, and it rotates as it moves, because "up" in a fisheye points at the centre of the circle, not at the ceiling.

In [ ]:
def place_bird(img, bird, t):
    b = room_dir(*flight_path(t))
    v = room_dir(*flight_path(t + 1 / FPS)) - room_dir(*flight_path(t - 1 / FPS))
    u = v - b * (v @ b); u /= np.linalg.norm(u)                    # heading
    up = np.array([0, 0, 1], np.float32)
    w = up - b * (up @ b) - u * (up @ u); w /= np.linalg.norm(w)   # card up, zero roll
    ch, cw = bird.shape[:2]
    hw = math.tan(math.radians(BIRD_DEG / 2)); hh = hw * ch / cw
    cos_r = math.cos(math.atan(math.hypot(hw, hh)) + 0.02)
    near = (ROOM @ b > cos_r) & INSIDE                             # only the pixels near the card
    if not near.any():
        return img
    ys, xs = np.where(near)
    ya, yb, xa, xb = ys.min(), ys.max() + 1, xs.min(), xs.max() + 1
    d = ROOM[ya:yb, xa:xb]
    p = d / np.maximum(d @ b, 1e-4)[..., None]                     # ray meets the card plane
    mx = ((p @ u) / hw * 0.5 + 0.5) * cw - 0.5
    my = (-(p @ w) / hh * 0.5 + 0.5) * ch - 0.5
    pre = bird.copy(); pre[..., :3] *= pre[..., 3:4]               # premultiply: clean edges
    s = cv2.remap(pre, mx.astype(np.float32), my.astype(np.float32),
                  cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=0)
    img[ya:yb, xa:xb] = img[ya:yb, xa:xb] * (1 - s[..., 3:4]) + s[..., :3]
    return img


def master_frame(rgb, t):
    img = place_bird(SKY.copy(), key(rgb[y0:y1, x0:x1]), t)
    img[~INSIDE] = 0
    return (np.clip(img, 0, 1) * 255).astype(np.uint8)

## Look at four moments of the lap

Front of the room at the bottom. At 0 s it is in front and to the left, at 7.5 s in front and to the right; at 15 s and 22.5 s it is behind the audience, higher up, near the top of the circle.

In [ ]:
times = [0.0, 7.5, 15.0, 22.5]
grid = Image.new("RGB", (1024, 1024))
src = list(frames(int(times[-1] * FPS) + 1))
for k, t in enumerate(times):
    m = Image.fromarray(master_frame(src[int(t * FPS)], t)).resize((512, 512))
    grid.paste(m, ((k % 2) * 512, (k // 2) * 512))
display(grid)

## Render the whole piece

900 frames, piped straight into ffmpeg with the encode settings from the delivery spec. A few minutes at 1024², about four times longer at 2048². Set `RENDER = True` to run it.

In [ ]:
RENDER = False

if RENDER:
    mp4 = f"flight_{RES}x{RES}_30fps.mp4"
    enc = subprocess.Popen(["ffmpeg", "-v", "error", "-y", "-f", "rawvideo", "-pix_fmt", "rgb24",
                            "-s", f"{RES}x{RES}", "-r", str(FPS), "-i", "-",
                            "-c:v", "libx264", "-profile:v", "high", "-pix_fmt", "yuv420p",
                            "-b:v", "60M", "-maxrate", "60M", "-bufsize", "120M", "-r", str(FPS), mp4],
                           stdin=subprocess.PIPE)
    for i, rgb in enumerate(frames(FPS * SECONDS)):
        enc.stdin.write(master_frame(rgb, i / FPS).tobytes())
        if i % 150 == 0:
            print(f"frame {i + 1}/{FPS * SECONDS}")
    enc.stdin.close(); enc.wait()
    print("done:", mp4)
    try:
        from google.colab import files
        files.download(mp4)
    except ImportError:
        pass

## Try changing

- `flight_path()`: keep the subject in front only, for example `az = 50 * math.sin(2 * math.pi * t / 20)`. At the turning points the card has to point straight up or down for a moment; fix it by easing the elevation, or by flipping the card instead of turning it.
- `BIRD_DEG`: the card's angular width. Above about 40° the subject starts to feel in your face.
- `make_sky()`: swap the dusk for footage. A 360° equirectangular video works as the background if you look it up by room azimuth and elevation, the way the stars are placed.
- Your own clip, shot on a green or blue sheet. For blue, swap `g` and `b` in `key()`.

## What usually breaks

| Symptom | Cause |
|---|---|
| A green fringe round the subject | No despill, or the ramp in `key()` starts too high for your backing. Light the backing evenly |
| Holes in the subject | Green in the subject itself (clothes, reflections), or the ramp is too wide |
| The subject stretches and spins near the edge of the circle | It was pasted onto the master instead of hung on a card in the room |
| It flies out of the dome behind the audience | Elevation below about 36° behind. Keep it higher there |
| The horizon tips as it flies | The card's up isn't kept towards the room's up: that is roll |